In [1]:
library(tidyverse)
library(vegan)
# set strings as factors to false
options(stringsAsFactors = FALSE)

── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: permute

Lade nötiges Paket: lattice

This is vegan 2.6-6.1



# read Wind data

In [17]:
wind_ds <- readRDS("processed/ERA5_data.rds")
wind_ds$date = as.Date(wind_ds$time)
wind_ds$time_month = format(wind_ds$date, format="%m-%Y")
wind_ds <- wind_ds %>% select(-date, -time)
str(wind_ds)#$date

'data.frame':	1012 obs. of  13 variables:
 $ tauoc     : num  0.944 0.96 0.955 0.958 0.982 ...
 $ sst       : num  299 298 298 299 299 ...
 $ sp        : num  101164 101103 101012 101065 101025 ...
 $ u10       : num  -5.9 -5.31 -5.31 -4.51 -3.32 ...
 $ v10       : num  -4.08 -4.4 -4.64 -3.61 -2.55 ...
 $ lsm       : num  0.0666 0.0666 0.0666 0.0666 0.0666 ...
 $ si10      : num  7.41 7.28 7.45 6.15 4.96 ...
 $ ewss      : num  -7883 -7164 -7379 -5068 -3336 ...
 $ e         : num  -0.00401 -0.0024 -0.00295 -0.00207 -0.0019 ...
 $ ro        : num  0.00 0.00 0.00 1.14e-06 3.05e-06 ...
 $ tp        : num  6.33e-05 1.05e-05 7.30e-05 8.09e-04 1.01e-03 ...
 $ mtpr      : num  7.45e-07 1.13e-07 8.58e-07 9.36e-06 1.17e-05 ...
 $ time_month: chr  "12-1939" "01-1940" "02-1940" "03-1940" ...


# read NISKIN data

In [7]:
niskin_ds <- readRDS("processed/Niskin_qchecked_100m.rds")
niskin_ds$time_month = format(niskin_ds$date, format="%m-%Y")
niskin_ds <- niskin_ds %>% select(-date) 
str(niskin_ds)

'data.frame':	229 obs. of  22 variables:
 $ O2_umol_kg         : num  178 150 150 145 150 ...
 $ O2_ml_L            : num  4.08 3.45 3.43 3.33 3.45 ...
 $ NO3_UDO            : num  0.949 3.035 7.475 6.715 6.414 ...
 $ PO4_UDO            : num  0.0625 0.2067 0.2984 0.317 0.3282 ...
 $ SiO4_UDO           : num  2.15 3.84 4.78 3.99 5.1 ...
 $ NH4_USF            : num  NA NA NA NA NA NA NA NA NA NA ...
 $ NO2_USF            : num  NA NA NA NA NA NA NA NA NA NA ...
 $ NO3_NO2_USF        : num  NA NA NA NA NA NA NA NA NA NA ...
 $ NO3_USF            : num  NA NA NA NA NA NA NA NA NA NA ...
 $ PO4_USF            : num  NA NA NA NA NA NA NA NA NA NA ...
 $ SiO4_USF           : num  NA NA NA NA NA NA NA NA NA NA ...
 $ NO3_merged         : num  0.949 3.035 7.475 6.715 6.414 ...
 $ PO4_merged         : num  0.0625 0.2067 0.2984 0.317 0.3282 ...
 $ SiO4_merged        : num  2.15 3.84 4.78 3.99 5.1 ...
 $ pH_corrected       : num  NA 8.01 7.99 7.97 NA ...
 $ Salinity_bottles   : num  36.8 NA 36.9 

# read HPLC data

In [11]:
hplc_ds <- readRDS("processed/HPLC_interpolated_sizes.rds")
hplc_ds$time_month = format(hplc_ds$date, format="%m-%Y")
hplc_ds <- hplc_ds %>% select(-date)
str(hplc_ds)

'data.frame':	174 obs. of  20 variables:
 $ source    : chr  "BBRS" "BBRS" "BBRS" "BBRS" ...
 $ Pras      : num  0.00467 0.00698 NA NA NA ...
 $ Lut       : num  NA NA NA NA NA ...
 $ Fuco      : num  0.0154 0.2567 0.2122 0.2912 1.2477 ...
 $ Perid     : num  0.00482 0.05636 0.01284 0.04261 0.03304 ...
 $ Allo      : num  NA 0.00432 0.00489 0.00669 NA ...
 $ But_fuco  : num  0.00642 0.00851 0.00381 0.00158 0.03133 ...
 $ Hex_fuco  : num  0.0199 0.0198 0.0221 0.0184 0.2189 ...
 $ Zea       : num  0.00697 0.00565 0.00578 0.0071 0.0166 ...
 $ Tot_Chl_b : num  0.0497 0.0519 0.0375 0.0248 0.0849 ...
 $ DP        : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Tot_Chl_a : num  0.127 0.434 0.278 0.28 2.243 ...
 $ TChl      : num  NA NA NA NA NA NA NA NA NA NA ...
 $ Chl_c1c2  : num  0.006 0.0959 0.0338 0.1235 0.0723 ...
 $ Chl_c3    : num  NA 0.04015 0.00811 NA 0.01714 ...
 $ DP2       : num  NA 0.529 0.392 0.53 NA ...
 $ micro     : num  NA 0.834 0.809 0.888 NA ...
 $ nano      : num  NA 0.0581 

## add pigment diversity measures

In [12]:
pigment_ds <- hplc_ds %>% select(time_month,Lut,Pras,Fuco,Perid,Allo,But_fuco,Hex_fuco,Zea,Tot_Chl_b,Tot_Chl_a,Chl_c1c2,Chl_c3) %>%

    filter(rowSums(is.na(.)) <6) %>% replace(is.na(.), 0)

Shannon_pigment <- diversity(pigment_ds[-1])
Pielou_pig <- Shannon_pigment/log(specnumber(pigment_ds[-1]))
PigmentRichness <- apply(pigment_ds[-1]>0,1,sum)
#Jaccard_pig <- vegdist(pigment_ds[-1],method="jaccard",binary=TRUE)


Pigment_diversity = cbind(pigment_ds[1],Shannon_pigment,Pielou_pig,PigmentRichness)

# read Zooplankton data

In [13]:
zoo_ds <- readRDS("processed/Zoo_dryBiomass.rds")
zoo_ds$time_month = format(zoo_ds$date, format="%m-%Y")
zoo_ds <- zoo_ds %>% select(-date)
str(zoo_ds)

tibble [153 × 3] (S3: tbl_df/tbl/data.frame)
 $ Mesh200   : num [1:153] 26.83 7.43 5.6 10.24 12.92 ...
 $ Mesh500   : num [1:153] 15.947 2.337 0.714 4.93 NA ...
 $ time_month: chr [1:153] "10-2001" "11-2001" "12-2001" "01-2002" ...


# read CTD data

In [14]:
ctd_ds <- readRDS("processed/CTD_Isotherm21_MLD.rds")


ctd_ds$time_month = format(ctd_ds$date, format="%m-%Y")

# in 2012-11 there were two measuerments, so I need to average these two:
ctd_ds2 <- ctd_ds %>% group_by(time_month) %>% 
    summarize(Isotherm_21 = mean(Isotherm_21), MLD= mean(MLD)) %>% ungroup()


In [15]:
str(ctd_ds2)

tibble [228 × 3] (S3: tbl_df/tbl/data.frame)
 $ time_month : chr [1:228] "01-1996" "01-1997" "01-1998" "01-1999" ...
 $ Isotherm_21: num [1:228] 103 64 91 76 82 92 123 57 83 85 ...
 $ MLD        : num [1:228] 39 26 21 22 37 31 37 29 38 27 ...


# MERGE DATA

In [18]:
str(wind_ds)

'data.frame':	1012 obs. of  13 variables:
 $ tauoc     : num  0.944 0.96 0.955 0.958 0.982 ...
 $ sst       : num  299 298 298 299 299 ...
 $ sp        : num  101164 101103 101012 101065 101025 ...
 $ u10       : num  -5.9 -5.31 -5.31 -4.51 -3.32 ...
 $ v10       : num  -4.08 -4.4 -4.64 -3.61 -2.55 ...
 $ lsm       : num  0.0666 0.0666 0.0666 0.0666 0.0666 ...
 $ si10      : num  7.41 7.28 7.45 6.15 4.96 ...
 $ ewss      : num  -7883 -7164 -7379 -5068 -3336 ...
 $ e         : num  -0.00401 -0.0024 -0.00295 -0.00207 -0.0019 ...
 $ ro        : num  0.00 0.00 0.00 1.14e-06 3.05e-06 ...
 $ tp        : num  6.33e-05 1.05e-05 7.30e-05 8.09e-04 1.01e-03 ...
 $ mtpr      : num  7.45e-07 1.13e-07 8.58e-07 9.36e-06 1.17e-05 ...
 $ time_month: chr  "12-1939" "01-1940" "02-1940" "03-1940" ...


In [20]:
CARIACO_dat_joined <- list(wind_ds, 
                           niskin_ds, 
                           hplc_ds,
                           zoo_ds,
                           ctd_ds2
                          ) %>% 
  reduce(full_join, by = c("time_month"))

In [21]:
CARIACO_dat_joined_truncated <- CARIACO_dat_joined[which(CARIACO_dat_joined$time_month=="11-1995"):which(CARIACO_dat_joined$time_month=="01-2017"),]

In [22]:
names(CARIACO_dat_joined_truncated)

[1] "tauoc"               "sst"                 "sp"                 
 [4] "u10"                 "v10"                 "lsm"                
 [7] "si10"                "ewss"                "e"                  
[10] "ro"                  "tp"                  "mtpr"               
[13] "time_month"          "O2_umol_kg"          "O2_ml_L"            
[16] "NO3_UDO"             "PO4_UDO"             "SiO4_UDO"           
[19] "NH4_USF"             "NO2_USF"             "NO3_NO2_USF"        
[22] "NO3_USF"             "PO4_USF"             "SiO4_USF"           
[25] "NO3_merged"          "PO4_merged"          "SiO4_merged"        
[28] "pH_corrected"        "Salinity_bottles"    "Temperature"        
[31] "Sigma_t"             "PrimaryProductivity" "Chlorophyll"        
[34] "Phaeopigments"       "source"              "Pras"               
[37] "Lut"                 "Fuco"                "Perid"              
[40] "Allo"                "But_fuco"            "Hex_fuco"           
[43] "Zea"                 "Tot_Chl_b"           "DP"                 
[46] "Tot_Chl_a"           "TChl"                "Chl_c1c2"           
[49] "Chl_c3"              "DP2"                 "micro"              
[52] "nano"                "pico"                "Mesh200"            
[55] "Mesh500"             "Isotherm_21"         "MLD"

In [23]:
saveRDS(CARIACO_dat_joined_truncated, "processed/CARIACO_EnvData_combined.rds")

# create RAW dataframe of Niskin and Phyto with Depth, not interpolated

In [1]:
phyto_raw <- readRDS("processed/PhytoplanktonRawCounts.RDS")
niskin_raw <- readRDS("processed/Niskin_RAW.RDS")

In [3]:
head(phyto_raw)
head(niskin_raw)

date,depth,Acanthoica,Akashiwo,Alexandrium,Algirosphaera,Amphidinium,Amphora,Anabaena,Anacystis,⋯,Hillea,Microcystis,Pandorina,Scenedesmus,Schuettiella,Spirulina,Trachelomonas,Climacosphenia,Coronosphaera,Gloeocapsa
<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1995-11-08 07:36:00,1,0,0,0,0,0,0,0,0.000,⋯,0,0,0,0,0,0,0,0,0,0
1995-11-08 07:36:00,7,0,0,0,0,0,0,0,0.000,⋯,0,0,0,0,0,0,0,0,0,0
1995-11-08 07:36:00,15,0,0,0,0,0,0,0,0.000,⋯,0,0,0,0,0,0,0,0,0,0
1995-11-08 07:36:00,25,0,0,0,0,0,0,0,0.000,⋯,0,0,0,0,0,0,0,0,0,0
1995-11-08 07:36:00,55,0,0,0,0,0,0,0,0.053,⋯,0,0,0,0,0,0,0,0,0,0
1995-11-08 07:36:00,75,0,0,0,0,0,0,0,0.000,⋯,0,0,0,0,0,0,0,0,0,0


,date,depth,O2_umol_kg,O2_ml_L,NO3_UDO,PO4_UDO,SiO4_UDO,NH4_USF,NO2_USF,NO3_NO2_USF,⋯,NO3_merged,PO4_merged,SiO4_merged,pH_corrected,Salinity_bottles,Temperature,Sigma_t,PrimaryProductivity,Chlorophyll,Phaeopigments
,<date>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1995-11-08,1,211.614,4.85,0.18,0.00,2.4,NA,NA,NA,⋯,0.18,0.00,2.4,NA,36.5760,27.49,23.720,NA,0.0940762,0.09
2,1995-11-08,7,192.391,4.41,0.17,0.00,2.8,NA,NA,NA,⋯,0.17,0.00,2.8,NA,36.5770,27.50,23.720,NA,0.0746952,0.06
3,1995-11-08,15,191.075,4.38,0.16,0.00,2.2,NA,NA,NA,⋯,0.16,0.00,2.2,NA,36.5900,27.49,23.760,NA,0.1000480,0.08
4,1995-11-08,25,190.565,4.37,0.17,0.00,NA,NA,NA,NA,⋯,0.17,0.00,NaN,NA,36.6118,26.30,24.156,NA,0.1180140,0.14
5,1995-11-08,35,186.131,4.27,0.85,0.01,1.9,NA,NA,NA,⋯,0.85,0.01,1.9,NA,36.7440,25.32,24.563,NA,0.1314760,0.20
6,1995-11-08,55,172.142,3.95,0.63,0.06,1.7,NA,NA,NA,⋯,0.63,0.06,1.7,NA,36.8290,24.75,24.800,NA,0.4180000,0.60


In [4]:
unique(phyto_raw$depth)

[1]   1   7  15  25  55  75 100

In [5]:
unique(niskin_raw$depth)

[1]    1    7   15   25   35   55   75  100  150  200  250  300  350  400  450
[16]  500  750 1000 1200  130  160 1310  275  233  230  260  220  115  140  170
[31]  180  270  310  325  320  225  280  335  295  290  375 1250 1228 1100 1160
[46] 1300   18  240  265  317  268  212  305   90  114  135  238  222  235  269
[61]  261  730  285  241  236  272  242  253 1259  259  256  266  267 1320  262
[76]  271  276  258  246 1220 1175  263  282  511 1275  257  245  281  254   NA
[91]  289

In [6]:
# filter uneccesary depths out of niskin
niskin_raw_filtdep <- niskin_raw %>% filter(!depth>100)

In [8]:
# from phyto_raw calculate div metrics per depth and date

# remove nanoflagellates from dataset (only identified to Genus level used)
#phyto_raw_filt <- phyto_raw %>% select(-`1`)

In [11]:
phyto_raw_filt_rm0 = phyto_raw[rowSums(phyto_raw[, c(-1,-2)])>0, ]

In [12]:
dim(phyto_raw)

[1] 1547  227

In [13]:
dim(phyto_raw_filt_rm0)

[1] 1420  227

In [14]:
raw_Shannon_gen <- diversity(phyto_raw_filt_rm0[c(-1,-2)])
raw_Pielou_gen <- raw_Shannon_gen/log(specnumber(phyto_raw_filt_rm0[c(-1,-2)]))
raw_GenusRichness <- apply(phyto_raw_filt_rm0[c(-1,-2)]>0,1,sum)

In [15]:
raw_diversity_ds = data.frame("date"=phyto_raw_filt_rm0$date, "depth"=phyto_raw_filt_rm0$depth, raw_GenusRichness, raw_Shannon_gen, raw_Pielou_gen)

In [17]:
merged_div_raw <- merge(niskin_raw_filtdep, raw_diversity_ds, by=c("date","depth"))
merged_phytonisk_raw <- merge(niskin_raw_filtdep, phyto_raw, by=c("date","depth"))

In [19]:
# remove nanoflagellates from dataset (only identified to Genus level used)
head(merged_div_raw)

,date,depth,O2_umol_kg,O2_ml_L,NO3_UDO,PO4_UDO,SiO4_UDO,NH4_USF,NO2_USF,NO3_NO2_USF,⋯,pH_corrected,Salinity_bottles,Temperature,Sigma_t,PrimaryProductivity,Chlorophyll,Phaeopigments,raw_GenusRichness,raw_Shannon_gen,raw_Pielou_gen
,<date>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
1,1995-11-08,1,211.614,4.85,0.18,0.00,2.4,NA,NA,NA,⋯,NA,36.5760,27.49,23.720,NA,0.0940762,0.09,19,1.8963552,0.6440464
2,1995-11-08,100,158.175,3.63,2.52,0.20,2.4,NA,NA,NA,⋯,NA,36.8980,23.97,24.937,NA,0.0680952,0.16,6,0.3054317,0.1704647
3,1995-11-08,15,191.075,4.38,0.16,0.00,2.2,NA,NA,NA,⋯,NA,36.5900,27.49,23.760,NA,0.1000480,0.08,16,2.4135491,0.8705038
4,1995-11-08,25,190.565,4.37,0.17,0.00,NA,NA,NA,NA,⋯,NA,36.6118,26.30,24.156,NA,0.1180140,0.14,18,2.1929912,0.7587229
5,1995-11-08,55,172.142,3.95,0.63,0.06,1.7,NA,NA,NA,⋯,NA,36.8290,24.75,24.800,NA,0.4180000,0.60,22,1.7162514,0.5552339
6,1995-11-08,7,192.391,4.41,0.17,0.00,2.8,NA,NA,NA,⋯,NA,36.5770,27.50,23.720,NA,0.0746952,0.06,21,1.9285108,0.6334362


In [20]:
saveRDS(merged_div_raw, "processed/Diversity_rawdepths.rds")

saveRDS(merged_phytonisk_raw, "processed/Phyto_niskin_rawdepths.rds")